[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/10_Deployment/03_Mobile_Deployment/Mobile_Deployment_Deep_Dive.ipynb)

# Mobile Deployment with ONNX Runtime — Deep Dive

A comprehensive treatment of deploying ONNX models on mobile platforms:
model size optimization, NNAPI/CoreML delegation, INT8 inference on ARM NEON,
battery-aware scheduling, and cross-platform frameworks.

---

## Table of Contents

| # | Section | Key Topics |
|---|---------|------------|
| 1 | [Mobile ML Landscape](#1) | Platform constraints, hardware diversity |
| 2 | [Model Size Budget](#2) | Download size, memory mapping, compression |
| 3 | [ARM NEON and INT8 Inference](#3) | SIMD operations, quantized matmul |
| 4 | [NNAPI Execution Provider](#4) | Android NPU delegation, graph partitioning |
| 5 | [CoreML Execution Provider](#5) | Apple Neural Engine, Metal performance |
| 6 | [Battery and Thermal Management](#6) | Power-aware scheduling, duty cycling |
| 7 | [Memory Management](#7) | Arena allocation, buffer reuse, mmap |
| 8 | [Cross-Platform Frameworks](#8) | Flutter, React Native, KMM integration |
| 9 | [Model Optimization Pipeline](#9) | Pruning, distillation, NAS for mobile |
| 10 | [Testing and Device Coverage](#10) | Fragmentation, CI on device farms |

---

<a id='1'></a>
## 1. Mobile ML Landscape

Mobile inference operates under the most stringent constraints of any deployment target. Unlike edge devices (single-purpose, controlled environment), mobile apps share resources with dozens of competing processes on heterogeneous hardware.

### The Mobile Constraint Envelope

```
┌──────────────────────────────────────────────────────────────────┐
│                    Mobile Device Constraint Stack                  │
├──────────────────────────────────────────────────────────────────┤
│  App Store Limits     │ APK < 150 MB, IPA < 200 MB (w/o ODR)    │
├──────────────────────────────────────────────────────────────────┤
│  RAM Budget           │ App < 200-400 MB (before OS kills it)    │
├──────────────────────────────────────────────────────────────────┤
│  Thermal Envelope     │ Sustained < 3-5W (phone), < 8W (tablet) │
├──────────────────────────────────────────────────────────────────┤
│  Battery Impact       │ < 2% drain per hour (background ML)     │
├──────────────────────────────────────────────────────────────────┤
│  Latency Budget       │ 16ms (60fps) to 100ms (perception)      │
├──────────────────────────────────────────────────────────────────┤
│  Startup Time         │ Model load < 500ms (first inference)     │
└──────────────────────────────────────────────────────────────────┘
```

### Hardware Landscape (2024)

| Platform | CPU | NPU/Accelerator | Peak INT8 TOPS |
|----------|-----|-----------------|----------------|
| Snapdragon 8 Gen 3 | Cortex-X4 + A720 | Hexagon NPU | 45 TOPS |
| Apple A17 Pro | P-cores + E-cores | Neural Engine (16-core) | 35 TOPS |
| Google Tensor G3 | Cortex-X3 + A715 | Edge TPU + custom ML | 30 TOPS |
| MediaTek Dimensity 9300 | Cortex-X4 | APU 7.0 | 37 TOPS |
| Samsung Exynos 2400 | Cortex-X4 + A720 | Dual-core NPU | 34 TOPS |

**Critical insight:** The NPU is 10-50× more power-efficient than CPU/GPU for supported operations. The entire mobile ML optimization story is about maximizing NPU delegation.

### Model Size Constraints

The total app download size constraint:

$$S_{\text{app}} = S_{\text{code}} + S_{\text{assets}} + S_{\text{model}} + S_{\text{runtime}} \leq S_{\text{store\_limit}}$$

For a typical app with ORT Mobile:

$$S_{\text{model}} \leq S_{\text{store\_limit}} - S_{\text{code}} - S_{\text{assets}} - S_{\text{ORT}}$$
$$S_{\text{model}} \leq 150\text{MB} - 20\text{MB} - 30\text{MB} - 8\text{MB} = 92\text{MB}$$

In practice, models should be much smaller (5-30 MB) to maintain fast downloads and good user experience.

<a id='2'></a>
## 2. Model Size Budget Mathematics

### Download Size vs Runtime Memory

These are different concerns:

$$S_{\text{download}} = S_{\text{model\_file}} \quad \text{(compressed in store)}$$

$$M_{\text{runtime}} = M_{\text{params}} + M_{\text{activations}} + M_{\text{runtime\_overhead}}$$

A model can be small on disk but large in memory (sparse models, compressed weights) or vice versa.

### Model Size Breakdown by Architecture

| Model | Params | FP32 Size | INT8 Size | Download (gzip) |
|-------|--------|-----------|-----------|------------------|
| MobileNetV3-Small | 2.5M | 10 MB | 2.5 MB | 2.1 MB |
| MobileNetV3-Large | 5.4M | 21.6 MB | 5.4 MB | 4.6 MB |
| EfficientNet-Lite0 | 4.7M | 18.8 MB | 4.7 MB | 4.0 MB |
| MobileBERT | 25M | 100 MB | 25 MB | 21 MB |
| YOLOv8n | 3.2M | 12.8 MB | 3.2 MB | 2.7 MB |

### Compression Analysis

ONNX models compress well because weight tensors have low entropy:

$$\text{Compression Ratio} = \frac{S_{\text{original}}}{S_{\text{compressed}}} = \frac{H_{\text{max}}}{H_{\text{actual}}}$$

For quantized INT8 weights (near-Gaussian distribution):

$$H(W_{\text{INT8}}) \approx \frac{1}{2}\log_2(2\pi e \sigma^2) \text{ bits}$$

With $\sigma \approx 30$ (typical for INT8 weights): $H \approx 7.4$ bits → minimal compression gain.

For FP32 weights: gzip typically achieves 15-25% compression because the mantissa bits appear random.

### On-Demand Resources (ODR)

For large models, use platform ODR mechanisms:

```
┌─────────────────────────────────────────────────┐
│  App Install (thin)                              │
│  ┌──────────────────┐                           │
│  │ Code + UI (20MB) │                           │
│  └──────────────────┘                           │
│           │                                      │
│           ▼  On first ML feature use            │
│  ┌──────────────────┐     ┌──────────────────┐  │
│  │  Download model  │◀────│  CDN / ODR       │  │
│  │  (background)    │     │  model-v2.onnx   │  │
│  └──────────────────┘     └──────────────────┘  │
│           │                                      │
│           ▼                                      │
│  ┌──────────────────┐                           │
│  │ Cache locally    │                           │
│  └──────────────────┘                           │
└─────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model size analysis for mobile deployment
models = {
    'MobileNetV3-S': {'params_M': 2.5, 'top1': 67.5},
    'MobileNetV3-L': {'params_M': 5.4, 'top1': 75.2},
    'EfficientNet-L0': {'params_M': 4.7, 'top1': 75.1},
    'EfficientNet-L4': {'params_M': 13.0, 'top1': 80.4},
    'MobileViT-S': {'params_M': 5.6, 'top1': 78.4},
    'MobileBERT': {'params_M': 25.0, 'top1': None},
    'YOLOv8n': {'params_M': 3.2, 'top1': None},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Model size comparison across precisions
names = list(models.keys())
params = [m['params_M'] for m in models.values()]
size_fp32 = [p * 4 for p in params]  # MB
size_fp16 = [p * 2 for p in params]
size_int8 = [p * 1 for p in params]
size_int4 = [p * 0.5 for p in params]

x = np.arange(len(names))
width = 0.2
axes[0].bar(x - 1.5*width, size_fp32, width, label='FP32', color='#e74c3c')
axes[0].bar(x - 0.5*width, size_fp16, width, label='FP16', color='#f39c12')
axes[0].bar(x + 0.5*width, size_int8, width, label='INT8', color='#2ecc71')
axes[0].bar(x + 1.5*width, size_int4, width, label='INT4', color='#3498db')

# Store limits
axes[0].axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50MB budget')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('Model Size (MB)')
axes[0].set_title('Model Size by Precision')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Download time analysis
bandwidths_mbps = [1, 5, 10, 25, 50]  # Mbps
model_sizes_mb = [5, 10, 25, 50, 100]  # MB

for bw in bandwidths_mbps:
    times = [s * 8 / bw for s in model_sizes_mb]  # seconds
    axes[1].plot(model_sizes_mb, times, 'o-', markersize=5, label=f'{bw} Mbps')

axes[1].axhline(y=3, color='green', linestyle='--', alpha=0.7, label='3s (good UX)')
axes[1].axhline(y=10, color='orange', linestyle='--', alpha=0.7, label='10s (tolerable)')
axes[1].set_xlabel('Model Size (MB)')
axes[1].set_ylabel('Download Time (seconds)')
axes[1].set_title('Model Download Time\n$t = S_{model} / B_{network}$')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 30)

# Accuracy vs size Pareto front (classification models only)
clf_models = {k: v for k, v in models.items() if v['top1'] is not None}
sizes = [v['params_M'] * 1 for v in clf_models.values()]  # INT8 size
accs = [v['top1'] for v in clf_models.values()]
names_clf = list(clf_models.keys())

axes[2].scatter(sizes, accs, s=100, c='#3498db', zorder=5)
for i, name in enumerate(names_clf):
    axes[2].annotate(name, (sizes[i], accs[i]), textcoords='offset points',
                     xytext=(5, 5), fontsize=8)
axes[2].set_xlabel('Model Size INT8 (MB)')
axes[2].set_ylabel('ImageNet Top-1 Accuracy (%)')
axes[2].set_title('Accuracy-Size Pareto Front\n(Mobile Classification Models)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mobile_model_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Mobile model size analysis complete.")
print(f"MobileNetV3-Small INT8: {2.5*1:.1f} MB - fits easily in any app")
print(f"MobileBERT INT8: {25*1:.0f} MB - requires careful size management")

<a id='3'></a>
## 3. ARM NEON and INT8 Inference

ARM NEON is a 128-bit SIMD extension available on all modern mobile ARM processors. It enables parallel processing of multiple integer/float values in a single instruction.

### NEON Register Layout for INT8

```
128-bit NEON register (Q register):
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│ b0 │ b1 │ b2 │ b3 │ b4 │ b5 │ b6 │ b7 │ b8 │ b9 │b10 │b11 │b12 │b13 │b14 │b15 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘
  16 × INT8 values processed in parallel

SDOT instruction (ARMv8.2+):
  Computes 4 dot products of 4-element INT8 vectors, accumulating into INT32:

  A: [a0 a1 a2 a3 | a4 a5 a6 a7 | a8 a9 a10 a11 | a12 a13 a14 a15]
  B: [b0 b1 b2 b3 | b4 b5 b6 b7 | b8 b9 b10 b11 | b12 b13 b14 b15]
  
  Result (4 × INT32):
  [Σ(a[0:3]·b[0:3]) | Σ(a[4:7]·b[4:7]) | Σ(a[8:11]·b[8:11]) | Σ(a[12:15]·b[12:15])]
```

### INT8 Quantized Matrix Multiplication

For quantized inference, the real-valued computation $Y = XW + b$ becomes:

$$Y_{\text{real}} = S_Y(Q_Y - Z_Y) = S_X S_W \cdot (Q_X - Z_X)(Q_W - Z_W) + b$$

Expanding:

$$Q_Y = \frac{S_X S_W}{S_Y} \left[ Q_X Q_W - Z_X \sum Q_W - Z_W \sum Q_X + N \cdot Z_X Z_W \right] + Z_Y + \frac{b}{S_Y}$$

The terms $Z_X \sum Q_W$ and $N \cdot Z_X Z_W$ can be **pre-computed** (they only depend on weights). This makes symmetric quantization ($Z = 0$) highly advantageous:

$$Q_Y = \frac{S_X S_W}{S_Y} \cdot Q_X Q_W + Z_Y + \frac{b}{S_Y} \quad \text{(symmetric, Z=0)}$$

### Throughput Calculation for ARM NEON

With SDOT instruction (4 INT8 MACs per element, 4 elements per instruction):

$$\text{TOPS}_{\text{INT8}} = \frac{N_{\text{exec\_units}} \times \text{MACs\_per\_cycle} \times f_{\text{clock}}}{10^{12}}$$

For a Cortex-A78 (2 NEON units, 16 INT8 MACs per cycle per unit, 3.0 GHz):

$$\text{TOPS} = \frac{2 \times 16 \times 3.0 \times 10^9}{10^{12}} = 0.096 \text{ TOPS per core}$$

With 4 big cores: $\approx 0.38$ TOPS on CPU alone.

Compare with NPU: 35-45 TOPS — **100× advantage** for supported operations.

### When INT8 CPU is Still Preferred

| Scenario | Why CPU INT8 wins |
|----------|-------------------|
| Unsupported ops on NPU | Fallback required |
| Very small models | NPU setup overhead > compute |
| Mixed precision needs | NPU may not support all dtypes |
| Determinism requirements | NPU numerics may vary across versions |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate INT8 quantized matrix multiplication
np.random.seed(42)

def quantize_tensor(tensor, n_bits=8, symmetric=True):
    """Quantize a floating-point tensor to INT8."""
    if symmetric:
        q_max = 2**(n_bits - 1) - 1
        q_min = -2**(n_bits - 1)
        abs_max = np.max(np.abs(tensor))
        scale = abs_max / q_max
        zero_point = 0
    else:
        q_max = 2**n_bits - 1
        q_min = 0
        t_min, t_max = tensor.min(), tensor.max()
        scale = (t_max - t_min) / (q_max - q_min)
        zero_point = int(np.round(q_min - t_min / scale))
    
    quantized = np.clip(np.round(tensor / scale) + zero_point, q_min, q_max).astype(np.int8)
    return quantized, scale, zero_point

def dequantize_tensor(quantized, scale, zero_point):
    """Dequantize INT8 tensor back to float."""
    return scale * (quantized.astype(np.float32) - zero_point)

# Simulate a typical mobile model layer: 256×256 matmul
M, K, N = 1, 256, 256  # Batch=1, typical FC layer

# Random weights and activations
W = np.random.randn(K, N).astype(np.float32) * 0.05
X = np.random.randn(M, K).astype(np.float32) * 0.1

# FP32 reference
Y_fp32 = X @ W

# INT8 quantized computation
X_q, X_scale, X_zp = quantize_tensor(X, 8, symmetric=True)
W_q, W_scale, W_zp = quantize_tensor(W, 8, symmetric=True)

# INT8 matmul (accumulate in INT32 as on NEON)
Y_int32 = X_q.astype(np.int32) @ W_q.astype(np.int32)

# Dequantize result
Y_scale = X_scale * W_scale
Y_int8 = (Y_int32 * Y_scale).astype(np.float32)

# Error analysis
error = np.abs(Y_fp32 - Y_int8)
relative_error = error / (np.abs(Y_fp32) + 1e-10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Weight distribution before/after quantization
axes[0, 0].hist(W.flatten(), bins=50, alpha=0.6, density=True, label='FP32', color='#3498db')
W_deq = dequantize_tensor(W_q, W_scale, W_zp)
axes[0, 0].hist(W_deq.flatten(), bins=50, alpha=0.6, density=True, label='Dequantized INT8', color='#e74c3c')
axes[0, 0].set_xlabel('Weight Value')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Weight Distribution: FP32 vs INT8')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Quantization error per output element
axes[0, 1].bar(range(min(50, N)), error[0, :50], color='#e74c3c', alpha=0.7)
axes[0, 1].set_xlabel('Output Index')
axes[0, 1].set_ylabel('Absolute Error')
axes[0, 1].set_title(f'INT8 Quantization Error (256×256 MatMul)\nMSE={np.mean(error**2):.2e}')
axes[0, 1].grid(True, alpha=0.3)

# Throughput comparison: CPU FP32 vs INT8 vs NPU
devices = ['CPU FP32', 'CPU INT8\n(NEON)', 'GPU FP16\n(Mali/Adreno)', 'NPU INT8\n(Hexagon)']
# Typical throughput for MobileNetV3 inference (ms)
latencies = [45, 18, 12, 3]
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

axes[1, 0].barh(devices, latencies, color=colors)
axes[1, 0].set_xlabel('Inference Latency (ms)')
axes[1, 0].set_title('MobileNetV3-Large Latency by Backend\n(Snapdragon 8 Gen 2)')
for i, v in enumerate(latencies):
    axes[1, 0].text(v + 0.5, i, f'{v}ms', va='center', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Power efficiency comparison
power_w = [3.0, 1.5, 2.5, 0.5]  # Watts during inference
energy_mj = [p * l for p, l in zip(power_w, latencies)]  # mJ per inference

axes[1, 1].bar(devices, energy_mj, color=colors, alpha=0.8)
axes[1, 1].set_ylabel('Energy per Inference (mJ)')
axes[1, 1].set_title('Energy Efficiency by Backend')
for i, v in enumerate(energy_mj):
    axes[1, 1].text(i, v + 1, f'{v:.0f}mJ', ha='center', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mobile_int8_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"INT8 Quantization Results (256×256 MatMul):")
print(f"  Max absolute error: {error.max():.6f}")
print(f"  Mean absolute error: {error.mean():.6f}")
print(f"  Relative MSE: {np.mean(relative_error**2):.2e}")
print(f"  X scale: {X_scale:.6f}, W scale: {W_scale:.6f}")

<a id='4'></a>
## 4. NNAPI Execution Provider (Android)

The Android Neural Networks API (NNAPI) provides a hardware-agnostic interface to on-device accelerators (NPU, DSP, GPU). ORT's NNAPI Execution Provider delegates supported subgraphs to NNAPI.

### NNAPI Delegation Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                    ONNX Runtime Mobile                           │
├─────────────────────────────────────────────────────────────────┤
│                    Graph Partitioner                             │
│  ┌───────────────────────┐  ┌──────────────────────────────┐   │
│  │  NNAPI-supported ops  │  │  CPU fallback ops             │   │
│  │  (Conv2D, ReLU,       │  │  (Unsupported custom ops,    │   │
│  │   DepthwiseConv,      │  │   dynamic shapes,            │   │
│  │   BatchNorm, Pool)    │  │   complex control flow)      │   │
│  └───────────┬───────────┘  └──────────────┬───────────────┘   │
│              │                              │                    │
│              ▼                              ▼                    │
│  ┌───────────────────────┐  ┌──────────────────────────────┐   │
│  │   NNAPI Runtime       │  │   ORT CPU Kernels            │   │
│  │   (Android system)    │  │   (XNNPACK / MLAS)           │   │
│  └───────────┬───────────┘  └──────────────────────────────┘   │
│              │                                                   │
│              ▼                                                   │
│  ┌───────────────────────┐                                      │
│  │ Hardware Accelerator  │                                      │
│  │ (Qualcomm Hexagon /   │                                      │
│  │  Samsung NPU /        │                                      │
│  │  Mali GPU)            │                                      │
│  └───────────────────────┘                                      │
└─────────────────────────────────────────────────────────────────┘
```

### Graph Partitioning Decision

A node is delegated to NNAPI if:

$$\text{delegate}(\text{node}) = \begin{cases}
\text{NNAPI} & \text{if op} \in \text{NNAPI\_supported\_ops} \land \text{dtypes match} \land \text{shapes static} \\
\text{CPU} & \text{otherwise}
\end{cases}$$

### Partition Quality Metric

$$\text{Delegation Ratio} = \frac{\text{FLOPs}_{\text{NNAPI}}}{\text{FLOPs}_{\text{total}}}$$

High delegation ratio (> 90%) is essential for performance. Each CPU↔NPU transition adds latency:

$$T_{\text{transition}} \approx 0.1\text{-}1.0 \text{ ms per boundary}$$

If a model has many small CPU segments between NNAPI segments, the transition overhead can negate NPU benefits.

### NNAPI Optimization Tips

| Tip | Rationale |
|-----|----------|
| Use INT8 quantized models | NPUs are optimized for INT8 |
| Avoid dynamic shapes | NNAPI requires static shapes |
| Prefer standard ops | Custom ops force CPU fallback |
| Fuse BatchNorm into Conv | Reduces graph complexity |
| Set NNAPI flags properly | `NNAPI_FLAG_USE_FP16` for GPU delegation |

<a id='5'></a>
## 5. CoreML Execution Provider (iOS)

Apple's CoreML framework provides access to the Neural Engine, GPU, and CPU on Apple Silicon. ORT's CoreML EP delegates supported operations to CoreML.

### Apple Silicon Neural Engine

```
┌─────────────────────────────────────────────────────────────┐
│                 Apple A17 Pro SoC                             │
├─────────────┬──────────────┬──────────────┬─────────────────┤
│  CPU        │  GPU         │Neural Engine │  Other          │
│  2P + 4E    │  6-core      │  16-core     │  ISP, Media     │
│  cores      │  Apple GPU   │  35 TOPS     │  Engines        │
│             │              │  INT8         │                 │
│  General    │  Parallel    │  ML-specific │  Camera,        │
│  compute    │  compute     │  accelerator │  Video          │
└─────────────┴──────────────┴──────────────┴─────────────────┘
         ▲              ▲              ▲
         │              │              │
         └──────────────┴──────────────┘
              Unified Memory (8-16 GB)
```

### CoreML EP Configuration

```python
providers = [
    ('CoreMLExecutionProvider', {
        'coreml_flags': 0,  # COREML_FLAG_USE_CPU_ONLY = 1
        'require_static_shape': True,
    }),
    'CPUExecutionProvider'
]
```

### Performance Tiers on Apple Silicon

| Compute Unit | Best For | Power | Latency Profile |
|-------------|----------|-------|------------------|
| Neural Engine | Large models, CNN/Transformer | Lowest | Fast, but compile overhead |
| GPU | Medium models, custom shaders | Medium | Good throughput |
| CPU (ANE unavail) | Small models, fallback | Highest | Predictable |

### FP16 on Apple Platforms

Apple Neural Engine natively operates in FP16. Converting models to FP16 is nearly free:

$$\text{Accuracy\_loss}_{\text{FP16}} \approx 0 \text{ (for most vision and NLP models)}$$

The exception: models with very large dynamic ranges in activations (rare in well-trained models).

$$\text{Representable range}_{\text{FP16}}: \pm 65504, \quad \epsilon_{\text{FP16}} = 2^{-10} \approx 9.77 \times 10^{-4}$$

<a id='6'></a>
## 6. Battery and Thermal Management

### Battery Drain Model

The battery impact of ML inference:

$$\Delta \text{Battery}(\%) = \frac{E_{\text{inference}} \cdot N_{\text{inferences}}}{E_{\text{battery}}} \times 100$$

where:

$$E_{\text{inference}} = P_{\text{active}} \cdot T_{\text{inference}}$$

For a typical phone battery (4500 mAh, 3.85V = 17.3 Wh = 62,280 J):

$$\text{Inferences per 1\% battery} = \frac{0.01 \times 62{,}280}{E_{\text{inference}}}$$

**Worked example:** MobileNetV3 on NPU (3ms @ 0.5W):

$$E_{\text{inference}} = 0.5 \times 0.003 = 1.5 \text{ mJ}$$
$$\text{Inferences per 1\% battery} = \frac{622.8}{0.0015} = 415{,}200$$

vs. same model on CPU FP32 (45ms @ 3.0W):

$$E_{\text{inference}} = 3.0 \times 0.045 = 135 \text{ mJ}$$
$$\text{Inferences per 1\% battery} = \frac{622.8}{0.135} = 4{,}613$$

**NPU is 90× more battery-efficient!**

### Thermal Throttling on Mobile

Mobile devices throttle aggressively because:
1. No active cooling (no fan)
2. Small thermal mass
3. User comfort (surface temperature < 45°C)

The sustained performance envelope:

$$P_{\text{sustained}} = \frac{\Delta T_{\text{skin\_limit}}}{R_{\theta,\text{junction-to-skin}}} \approx \frac{45 - 25}{5} = 4 \text{ W}$$

### Duty Cycling Strategy

For continuous inference (e.g., real-time camera), use duty cycling:

$$\text{duty\_cycle} = \frac{T_{\text{inference}}}{T_{\text{period}}} = \frac{T_{\text{inference}}}{T_{\text{inference}} + T_{\text{idle}}}$$

$$P_{\text{avg}} = P_{\text{active}} \cdot \text{duty\_cycle} + P_{\text{idle}} \cdot (1 - \text{duty\_cycle})$$

To stay within 4W sustained:

$$\text{duty\_cycle}_{\max} = \frac{P_{\text{sustained}} - P_{\text{idle}}}{P_{\text{active}} - P_{\text{idle}}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Battery and thermal analysis for mobile inference
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Battery drain comparison
backends = ['CPU FP32', 'CPU INT8\n(NEON)', 'GPU FP16', 'NPU INT8']
power_w = [3.0, 1.5, 2.5, 0.5]
latency_ms = [45, 18, 12, 3]
energy_mj = [p * l for p, l in zip(power_w, latency_ms)]

battery_wh = 17.3  # 4500mAh × 3.85V
battery_j = battery_wh * 3600

inferences_per_percent = [battery_j * 0.01 / (e/1000) for e in energy_mj]

axes[0, 0].bar(backends, [i/1000 for i in inferences_per_percent], color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'])
axes[0, 0].set_ylabel('Inferences per 1% Battery (thousands)')
axes[0, 0].set_title('Battery Efficiency by Backend\n(MobileNetV3, 4500mAh battery)')
axes[0, 0].grid(True, alpha=0.3)
for i, v in enumerate(inferences_per_percent):
    axes[0, 0].text(i, v/1000 + 5, f'{v/1000:.0f}K', ha='center', fontsize=9)

# Thermal simulation: sustained inference
t = np.linspace(0, 300, 1000)  # 5 minutes
T_ambient = 25
R_thermal = 5.0  # °C/W (phone without case)
C_thermal = 10.0  # J/°C (small thermal mass)
tau = R_thermal * C_thermal

T_skin_limit = 42  # User comfort limit
T_throttle = 40  # Start throttling before skin limit

for P, label, color in zip([4.0, 3.0, 1.5], 
                            ['CPU burst (4W)', 'GPU sustained (3W)', 'NPU (1.5W)'],
                            ['#e74c3c', '#f39c12', '#2ecc71']):
    T = T_ambient + P * R_thermal * (1 - np.exp(-t / tau))
    axes[0, 1].plot(t, T, linewidth=2, color=color, label=label)

axes[0, 1].axhline(y=T_throttle, color='orange', linestyle='--', alpha=0.7, label=f'Throttle ({T_throttle}°C)')
axes[0, 1].axhline(y=T_skin_limit, color='red', linestyle='--', alpha=0.7, label=f'Skin limit ({T_skin_limit}°C)')
axes[0, 1].set_xlabel('Time (seconds)')
axes[0, 1].set_ylabel('Skin Temperature (°C)')
axes[0, 1].set_title('Thermal Behavior: Sustained Inference')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Duty cycle analysis
duty_cycles = np.linspace(0.01, 1.0, 100)
P_active = 3.0
P_idle = 0.3
P_sustained_limit = 4.0

P_avg = P_active * duty_cycles + P_idle * (1 - duty_cycles)
max_duty = (P_sustained_limit - P_idle) / (P_active - P_idle)

# Battery life at different duty cycles
battery_hours = battery_wh / P_avg

axes[1, 0].plot(duty_cycles * 100, battery_hours, 'b-', linewidth=2)
axes[1, 0].axvline(x=max_duty*100, color='red', linestyle='--', 
                   label=f'Thermal limit ({max_duty*100:.0f}%)')
axes[1, 0].set_xlabel('Duty Cycle (%)')
axes[1, 0].set_ylabel('Battery Life (hours)')
axes[1, 0].set_title('Battery Life vs Inference Duty Cycle\n(CPU FP32 @ 3W active, 0.3W idle)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim(0, 100)

# FPS vs battery life tradeoff
fps_values = [1, 5, 10, 15, 30]
battery_hours_by_fps = []
for fps in fps_values:
    # Time spent inferring per second
    infer_time_per_sec = fps * 3 / 1000  # NPU: 3ms per frame
    dc = infer_time_per_sec
    p_avg = 0.5 * dc + 0.3 * (1 - dc)  # NPU power model
    battery_hours_by_fps.append(battery_wh / p_avg)

axes[1, 1].bar([str(f) for f in fps_values], battery_hours_by_fps, 
              color='#3498db', alpha=0.8)
axes[1, 1].set_xlabel('Target FPS')
axes[1, 1].set_ylabel('Battery Life (hours)')
axes[1, 1].set_title('Battery Life vs Target FPS\n(NPU INT8, MobileNetV3)')
axes[1, 1].grid(True, alpha=0.3)
for i, v in enumerate(battery_hours_by_fps):
    axes[1, 1].text(i, v + 0.5, f'{v:.1f}h', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('mobile_battery_thermal.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Max sustained duty cycle (CPU @ 3W): {max_duty*100:.0f}%")
print(f"NPU advantage: {inferences_per_percent[3]/inferences_per_percent[0]:.0f}× more efficient than CPU FP32")

<a id='7'></a>
## 7. Memory Management on Mobile

### Mobile Memory Hierarchy

```
┌─────────────────────────────────────────────────────┐
│  App Memory Budget (200-400 MB before OOM kill)      │
├─────────────────────────────────────────────────────┤
│                                                      │
│  ┌────────────────┐  ┌───────────────┐              │
│  │  ORT Session   │  │  App Code +   │              │
│  │  ┌──────────┐  │  │  UI Buffers   │              │
│  │  │ Weights  │  │  │  (50-100 MB)  │              │
│  │  │ (mmap'd) │  │  │               │              │
│  │  ├──────────┤  │  └───────────────┘              │
│  │  │Activation│  │                                  │
│  │  │  Arena   │  │  ┌───────────────┐              │
│  │  │ (reused) │  │  │  Camera/Image │              │
│  │  ├──────────┤  │  │  Buffers      │              │
│  │  │ I/O Bufs │  │  │  (10-50 MB)   │              │
│  │  └──────────┘  │  └───────────────┘              │
│  └────────────────┘                                  │
└─────────────────────────────────────────────────────┘
```

### Memory Mapping (mmap) for Weights

Instead of loading all weights into heap memory, mmap allows the OS to page weights on-demand:

$$M_{\text{RSS}} \ll M_{\text{file}} \quad \text{(only accessed pages resident)}$$

Benefits:
- Startup time: $T_{\text{load}} \approx 0$ (weights loaded on first access)
- Memory pressure: OS can evict unused weight pages
- Sharing: Multiple sessions can share the same physical pages

ORT Mobile supports external data format:
```
model.onnx          (graph structure, ~100 KB)
model.onnx.data     (weight tensors, mmap'd)
```

### Activation Arena Reuse

ORT's memory planner identifies non-overlapping lifetimes and reuses buffers:

$$M_{\text{arena}} = \max_t \sum_{i : \text{live}(i, t)} M_i \quad \ll \quad \sum_i M_i$$

For a typical CNN with sequential structure:

$$M_{\text{arena}} \approx 2 \times \max_l M_{\text{activation}}^{(l)}$$

(Only input and output of current layer need to be live simultaneously)

### Low-Memory Strategy

When `didReceiveMemoryWarning` (iOS) or `onTrimMemory` (Android) fires:

1. Release non-essential sessions
2. Clear inference result caches
3. If critical session needed, switch to smaller model variant
4. Never hold large tensors beyond their immediate use

<a id='8'></a>
## 8. Cross-Platform Frameworks

### Integration Patterns

```
┌─────────────────────────────────────────────────────────────────┐
│  Cross-Platform App                                              │
│                                                                   │
│  ┌─────────────────────────────────────────────────────────────┐ │
│  │  UI Layer (Dart / JS / Kotlin Multiplatform)                │ │
│  └──────────────────────────┬──────────────────────────────────┘ │
│                             │ Platform Channel / FFI              │
│  ┌──────────────────────────▼──────────────────────────────────┐ │
│  │  Native ML Module                                           │ │
│  │  ┌────────────────┐         ┌────────────────┐             │ │
│  │  │  Android (JNI) │         │  iOS (Swift)   │             │ │
│  │  │  ORT Mobile    │         │  ORT Mobile    │             │ │
│  │  │  NNAPI EP      │         │  CoreML EP     │             │ │
│  │  └────────────────┘         └────────────────┘             │ │
│  └─────────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────────┘
```

### Flutter Integration

```dart
// Platform channel approach
class OnnxInference {
  static const _channel = MethodChannel('onnx_inference');
  
  Future<List<double>> predict(Float32List input) async {
    final result = await _channel.invokeMethod('predict', {
      'input': input,
      'shape': [1, 3, 224, 224],
    });
    return (result as List).cast<double>();
  }
}
```

### React Native (Turbo Modules)

```typescript
// New Architecture: JSI-based Turbo Module
interface OnnxSpec extends TurboModule {
  loadModel(path: string): Promise<boolean>;
  predict(input: Float32Array, shape: number[]): Promise<Float32Array>;
}
```

### Key Anti-Pattern: Bridge Overhead

Never transfer large tensors repeatedly across the JS/Dart bridge:

$$T_{\text{bridge}} = T_{\text{serialize}} + T_{\text{copy}} + T_{\text{deserialize}} \propto N_{\text{elements}}$$

For a 224×224×3 float32 image: $N = 150{,}528$ elements = 602 KB per transfer.

**Solution:** Keep tensor allocation and inference native-side. Only transfer:
- Small result arrays (class probabilities, bounding boxes)
- Handles/references to native buffers

In [ ]:
import numpy as np
import onnxruntime as ort

# Mobile-optimized session configuration
print("=" * 60)
print("ORT Mobile Configuration Demo")
print("=" * 60)

# Session options for mobile
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = 2  # Conservative for battery
so.inter_op_num_threads = 1  # Minimize thread overhead
so.enable_mem_pattern = True  # Enable memory reuse
so.enable_cpu_mem_arena = True  # Pre-allocate buffers

# Platform-specific EP selection
available = ort.get_available_providers()
print(f"\nAvailable EPs: {available}")

# Simulate platform detection
import platform
system = platform.system()

if 'NnapiExecutionProvider' in available:
    providers = ['NnapiExecutionProvider', 'CPUExecutionProvider']
    print("Platform: Android - using NNAPI EP")
elif 'CoreMLExecutionProvider' in available:
    providers = ['CoreMLExecutionProvider', 'CPUExecutionProvider']
    print("Platform: iOS - using CoreML EP")
else:
    providers = ['CPUExecutionProvider']
    print(f"Platform: {system} - using CPU EP (mobile EPs not available)")

print(f"\nSession configuration:")
print(f"  Optimization level: ORT_ENABLE_ALL")
print(f"  Intra-op threads: {so.intra_op_num_threads}")
print(f"  Inter-op threads: {so.inter_op_num_threads}")
print(f"  Memory pattern: {so.enable_mem_pattern}")
print(f"  CPU arena: {so.enable_cpu_mem_arena}")
print(f"  Provider chain: {providers}")

# Memory estimation for common mobile models
print(f"\n{'Model':<20} {'Params':<10} {'INT8 Size':<12} {'Est. Runtime':<15}")
print("-" * 57)
mobile_models = [
    ('MobileNetV3-Small', 2.5, 2.5, 15),
    ('MobileNetV3-Large', 5.4, 5.4, 25),
    ('EfficientNet-L0', 4.7, 4.7, 22),
    ('YOLOv8n', 3.2, 3.2, 35),
    ('MobileBERT', 25.0, 25.0, 80),
]
for name, params, size_mb, runtime_mb in mobile_models:
    print(f"{name:<20} {params:<10.1f}M {size_mb:<12.1f}MB {runtime_mb:<15}MB")

<a id='9'></a>
## 9. Model Optimization Pipeline for Mobile

### Optimization Stages

```
┌───────────┐   ┌───────────┐   ┌───────────┐   ┌───────────┐   ┌───────────┐
│  Train    │──▶│  Prune    │──▶│  Export   │──▶│ Quantize  │──▶│  Deploy   │
│  (FP32)  │   │ (struct.) │   │  (ONNX)  │   │  (INT8)   │   │  (Mobile) │
└───────────┘   └───────────┘   └───────────┘   └───────────┘   └───────────┘
     100%           -30%            -0%            -75%           Final
   accuracy      channels       (format)        (precision)      model
```

### Structured Pruning

Remove entire channels based on importance score:

$$\text{importance}(c) = \|W[:, c, :, :]\|_1 \quad \text{(L1 norm of channel)}$$

Prune channels where:

$$\text{prune}(c) = \mathbb{1}\left[\text{importance}(c) < \text{threshold}_{p}\right]$$

After pruning with ratio $r$:

$$\text{FLOPs}_{\text{pruned}} \approx (1-r)^2 \cdot \text{FLOPs}_{\text{original}}$$

(Quadratic because both input and output channels are reduced)

### Knowledge Distillation for Mobile

Train a small student model to mimic a large teacher:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{CE}}(y, \hat{y}_{\text{student}}) + (1-\alpha) \cdot T^2 \cdot D_{KL}\left(\sigma\left(\frac{z_T}{T}\right) \| \sigma\left(\frac{z_S}{T}\right)\right)$$

where $T$ is temperature and $\alpha$ balances hard vs soft targets.

### Neural Architecture Search (NAS) for Mobile

MobileNet-style architectures are products of NAS with mobile constraints:

$$\text{maximize } \text{Accuracy}(\mathcal{A}) \quad \text{s.t.} \quad \text{Latency}(\mathcal{A}) \leq T_{\text{target}}$$

This is often reformulated as:

$$\text{maximize } \text{Accuracy}(\mathcal{A}) \cdot \left(\frac{\text{Latency}(\mathcal{A})}{T_{\text{target}}}\right)^{-\beta}$$

where $\beta$ controls the latency penalty strength.

<a id='10'></a>
## 10. Testing and Device Coverage

### The Fragmentation Problem

Android has thousands of device/SoC combinations. The same model may:
- Work on Snapdragon but fail on MediaTek (NNAPI driver differences)
- Produce different numerical results across GPU vendors
- Crash on older API levels (NNAPI version constraints)

### Testing Strategy

| Level | Scope | Tooling |
|-------|-------|--------|
| Unit | Op-level accuracy | Local, per-commit |
| Integration | Full model inference | CI with emulators |
| Device | Real hardware coverage | Firebase Test Lab / AWS Device Farm |
| Performance | Latency/memory on target | Device-specific benchmarks |

### Minimum Device Coverage Matrix

```
┌────────────────┬──────────────┬──────────────┬──────────────┐
│   SoC Family   │  Budget      │  Mid-range   │  Flagship    │
├────────────────┼──────────────┼──────────────┼──────────────┤
│ Qualcomm       │ SD 4 Gen 2   │ SD 7 Gen 3   │ SD 8 Gen 3   │
│ MediaTek       │ Dimensity 700│ Dimensity 8k │ Dimensity 9k │
│ Samsung        │ Exynos 1380  │ Exynos 2200  │ Exynos 2400  │
│ Apple          │ A15 (SE)     │ A16 (15)     │ A17 Pro      │
│ Google         │ Tensor G1    │ Tensor G2    │ Tensor G3    │
└────────────────┴──────────────┴──────────────┴──────────────┘
```

### Numerical Parity Testing

For each target device, validate:

$$\max_{i} |y_i^{\text{device}} - y_i^{\text{reference}}| < \epsilon_{\text{tolerance}}$$

Typical tolerances:
- FP32 CPU: $\epsilon = 10^{-5}$
- FP16 GPU: $\epsilon = 10^{-2}$
- INT8 NPU: $\epsilon = 10^{-1}$ (validate task-level metrics instead)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Mobile deployment decision framework
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Radar chart: backend comparison for mobile
categories = ['Latency', 'Battery\nEfficiency', 'Coverage\n(devices)', 
              'Accuracy', 'Startup\nTime', 'Memory']
N = len(categories)

backends_scores = {
    'CPU FP32': [0.3, 0.2, 1.0, 1.0, 0.9, 0.3],
    'CPU INT8 (NEON)': [0.5, 0.5, 1.0, 0.9, 0.9, 0.7],
    'NNAPI/CoreML': [0.9, 0.9, 0.6, 0.85, 0.5, 0.8],
    'GPU FP16': [0.7, 0.6, 0.7, 0.95, 0.6, 0.5],
}

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ax = plt.subplot(121, polar=True)
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
for (name, scores), color in zip(backends_scores.items(), colors):
    values = scores + scores[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Mobile Backend Comparison', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

# Decision flowchart as bar chart
ax2 = plt.subplot(122)
scenarios = ['Camera\n(real-time)', 'Photo\n(on-demand)', 'NLP\n(keyboard)', 
             'Background\n(analytics)', 'AR/VR\n(low-latency)']
recommended = ['NPU INT8', 'GPU FP16', 'CPU INT8', 'NPU INT8', 'GPU FP16']
latency_req = [16, 200, 100, 1000, 8]  # ms

colors_map = {'NPU INT8': '#2ecc71', 'GPU FP16': '#3498db', 'CPU INT8': '#f39c12'}
bar_colors = [colors_map[r] for r in recommended]

bars = ax2.bar(scenarios, latency_req, color=bar_colors, alpha=0.8)
ax2.set_ylabel('Latency Budget (ms)')
ax2.set_title('Recommended Backend by Use Case')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

for i, (bar, rec) in enumerate(zip(bars, recommended)):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
             rec, ha='center', fontsize=8, fontweight='bold')

# Custom legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors_map.items()]
ax2.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('mobile_decision_framework.png', dpi=150, bbox_inches='tight')
plt.show()
print("Mobile deployment decision framework complete.")

## Summary

Mobile deployment demands the most holistic optimization approach across all deployment targets. The key constraints and solutions:

### Core Equations

| Concept | Formula |
|---------|--------|
| **Size budget** | $S_{\text{model}} \leq S_{\text{store}} - S_{\text{code}} - S_{\text{assets}} - S_{\text{ORT}}$ |
| **Battery drain** | $\Delta\% = \frac{P \cdot T \cdot N}{E_{\text{battery}}} \times 100$ |
| **NPU throughput** | $\text{TOPS}_{\text{NPU}} \gg \text{TOPS}_{\text{CPU}}$ (100×) |
| **Thermal limit** | $P_{\text{sustained}} = \Delta T / R_{\theta}$ |
| **INT8 matmul** | $Q_Y = (S_X S_W / S_Y) \cdot Q_X Q_W + Z_Y + b/S_Y$ |

### Key Takeaways

1. **NPU delegation is the #1 performance lever** — 10-50× better than CPU, 90× more battery-efficient
2. **INT8 quantization unlocks NPU** — Most mobile NPUs only support INT8 natively
3. **Device fragmentation requires broad testing** — Same model, different behavior across SoCs
4. **Memory management is critical** — Mobile OS kills apps that exceed memory budgets
5. **Battery is a first-class constraint** — Users uninstall apps that drain battery

---

*Next: [Web Deployment](../04_Web_Deployment_ONNX_JS/Web_Deployment_Deep_Dive.ipynb) explores browser-based inference with WebAssembly and WebGPU.*